## Notebook Description: Data Curation and Averaging of Gas and Nutrition Data

This notebook outlines a comprehensive data curation and preprocessing workflow for two datasets: 'gas' and 'nutrition'. The primary objectives include standardizing categorical variables, filtering out erroneous or incomplete records, and aggregating replicate measurements. Key methodologies involve leveraging the pandas library for data manipulation, including the use of `df.replace()` for consistent categorization (e.g., 'Breding ' to 'Breeding'), `df.str.contains()` and `df.isna()` for identifying and marking rows for deletion based on specific keywords in remark columns or missing 'methane_intensity' values, and `df.groupby().mean()` for calculating average values and replicate counts (e.g., $\text{n_replicates_gas}$, $\text{n_replicates_nutrition}$) for each unique sample identifier ($\text{id_lab}$). The outcome is a set of cleaned and averaged datasets ready for further analysis, specifically $\text{gas_clean_av2}$ and $\text{nutrition_av2}$, which condense experimental replicates into single, representative entries.

# 1.0 Libraries

In [179]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [180]:
import numpy as np
import pandas as pd

# 2.0 Import data

In [181]:
gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_26_trial_database_curation/subsets_1_2_3_4_5_6_1h_gas_sorted_by_sql.csv')
nutrition = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_26_trial_database_curation/subsets_1_2_3_4_5_6_1h_nutrition_sorted_by_sql.csv')

In [182]:
gas.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
0,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,33.24064574,0,95.43754089,34.82973831,Standar,NaN,NaN,NaN,NaN,NaN
1,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,32.48546917,0,89.75200777,36.19469912,Standar,NaN,NaN,NaN,NaN,NaN


In [183]:
gas.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2'],
      dtype='object')

In [184]:
nutrition.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,1,1.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,91.930807,12.725691,87.274309,33.010522,28.489614,45.210329
1,1,2.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,92.010000,12.629062,87.370938,33.010522,27.612657,45.235119


# 3.0 Formatting

## 3.1 Drop duplicates

In [185]:
gas.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2'],
      dtype='object')

In [186]:
key = ['id_lab', 'batch', 'run', 'replication', 'syrange']

In [187]:
duplicates = gas[
    gas.duplicated(subset=key, keep=False)
].sort_values(key)

In [188]:
print(len(duplicates))
duplicates.iloc[:-10, :20]

2561


,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,batch,run,replication,syrange,sample_weight_g,undigested_dm_g,dm_incubated,digested_feed_mg
549,1,406,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,1,1.0,0.5002,0.2082,462.036651,253.836651
550,1,406,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,1,1.0,0.5002,0.2082,462.036651,253.836651
551,1,407,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,2,2.0,0.5001,0.2150,461.944280,246.944280
552,1,407,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,2,2.0,0.5001,0.2150,461.944280,246.944280
553,1,408,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,3,3.0,0.5,0.2217,461.851910,240.151910
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9785,5,1658,ISABEL MOLINA,F26-2902-(100%),STAR-GRASS-Porvenir,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,NaN,36,85,3,2,152.0,0.5,0.3334,462.450000,129.050000
9904,5,1775,LMF-invivo,F26-2902-(100%),START-GRASS,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,NaN,37,86,1,1,0.0,0.5002,0.3117,462.634980,150.934980
9905,5,1775,ISABEL MOLINA,F26-2902-(100%),STAR-GRASS-Porvenir,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,NaN,37,86,1,1,0.0,0.5002,0.3117,462.634980,150.934980
9906,5,1776,LMF-invivo,F26-2902-(100%),START-GRASS,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,NaN,37,86,1,2,0.0,0.5002,0.3060,462.634980,156.634980


## 3.2 Requisitioner update

### 3.2.1 Breeding and Genetic bank corrections

In [189]:
gas.requisitioner.unique()

array(['LMF', 'Genetic bank', 'Breeding', nan, 'Set 2', 'Breding ',
       'LMF-invivo', 'Benchmark', 'Genbank', 'Isabel Molina',
       'Mauricio Sotelo', 'Jacobo Arango/Alejandro Montoya',
       'Jacobo Arango/ Alejandro Montoya', 'Genebank', 'GENBANK',
       'BREEDING', 'ISABEL MOLINA',
       ' ISABEL MOLINA/ EXP Camaras In vivo ',
       'JACOBO ARANGO/ ALEJANDRO MONTOYA', 'JUAN CARDOSO'], dtype=object)

In [190]:
# Define the mapping for requisitioner
requisitioner_mapping = {
    'Breding ': 'Breeding',
    'BREEDING': 'Breeding',
    'Genbank': 'Genetic_bank',
    'GENBANK': 'Genetic_bank',
    'Genebank': 'Genetic_bank',
    'Genetic bank': 'Genetic_bank'
}

# Apply the mapping to centers datt
gas['requisitioner'] = gas['requisitioner'].replace(requisitioner_mapping)
nutrition['requisitioner'] = nutrition['requisitioner'].replace(requisitioner_mapping)

### 3.2.2 Genetic bank instead breeding for Herbaceous_legumes

In [191]:
gas.loc[gas["functional_group"] == "Herbaceous_legumes","requisitioner"] = "Genetic_bank"

In [192]:
nutrition.loc[nutrition["functional_group"] == "Herbaceous_legumes","requisitioner"] = "Genetic_bank"

## 3.3 Urocloa Taxonomy update


In [193]:
# Define the mapping for tax name
tax_name_mapping = {
    'Brachiaria humidicola': 'Urochloa humidicola',
    'Brachiaria interespecifico': 'Urochloa interespecific',
    'Brachiaria interespecifico ': 'Urochloa interespecific',
    'Urochloa interespecifico': 'Urochloa interespecific'

}

# Apply the mapping to centers datt
gas['tax_name'] = gas['tax_name'].replace(tax_name_mapping)
nutrition['tax_name'] = nutrition['tax_name'].replace(tax_name_mapping)

In [194]:
# Define the mapping for genus
genus_mapping = {
    'Brachiaria': 'Urochloa'
}

# Apply the mapping to centers datt
gas['genus'] = gas['genus'].replace(genus_mapping)
nutrition['genus'] = nutrition['genus'].replace(genus_mapping)

In [195]:
# Define the mapping for species
species_mapping = {
    'interespecifico': 'interespecific',
    ' interespecifico': 'interespecific',
}

# Apply the mapping to centers datt
gas['species'] = gas['species'].replace(species_mapping)
nutrition['species'] = nutrition['species'].replace(species_mapping)

## 3.4 Repeated samples

In [196]:
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1651"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1652"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1653"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1654"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1655"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1656"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1658"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1664"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1665"))]

## 3.5 Functional group update

Based on Juan José comments in 2026_07_16

In [197]:
herbaceous_legumes_to_shrub_trees = ['Aeschynomene americana', 'Aeschynomene sensitiva', 'Coursetia caribaea', 'Crotalaria juncea',
                                     'Crotalaria micans','Desmodium sequax', 'Indigofera suffruticosa']

gas.loc[gas["tax_name"].isin(herbaceous_legumes_to_shrub_trees), "functional_group"] = "Shrub_Trees"
nutrition.loc[nutrition["tax_name"].isin(herbaceous_legumes_to_shrub_trees), "functional_group"] = "Shrub_Trees"


In [198]:
shrub_trees_to_herbaceous_legumes =  ['Canavalia sp.']

gas.loc[gas["tax_name"].isin(shrub_trees_to_herbaceous_legumes), "functional_group"] = "Herbaceous_legumes"
nutrition.loc[nutrition["tax_name"].isin(shrub_trees_to_herbaceous_legumes), "functional_group"] = "Herbaceous_legumes"

## 3.6 Samples to delete

According with Alejandra review on first dashboard revision

In [199]:
to_delete = ['AMC-ICARDA-156302', 'AMC-ICARDA-165072', 'AMC-ICARDA-168125', 'CIAT-10647',
             'Sample-2', 'ILRI-7384-52-120169'  , 'Sample-4', 'Sample-5', 'Sample-3',
             'Muestra-#-02', 'Sample-7', 'Sample-1',

             'AMC-ICARDA-156302', 'ILRI-2273-27-119732', 'ILRI-13946-29', 'CIAT-17768',

             'ILRI-21682-24-119729', 'ILRI-15551'
             ]

In [200]:
gas = gas[~gas["id"].isin(to_delete)]
nutrition = nutrition[~nutrition["id"].isin(to_delete)]

Remove the sample of control 'Star grass' with the fewest replicates

In [201]:
sample_8 = gas[gas["id"] == "Sample-8"]
sample_8["id_lab"].value_counts()

,count
id_lab,
F25-0008,742
F25-0019,6


In [202]:
gas = gas[~gas["id_lab"].isin(["F25-0019"])]

In [203]:
sample_8 = gas[gas["id"] == "Sample-8"]
sample_8.id_lab.unique()

array(['F25-0008'], dtype=object)

In [204]:
nutrition = nutrition[~nutrition["id_lab"].isin(["F25-0019"])]

In [205]:
sample_8_nu = nutrition[nutrition["id"] == "Sample-8"]
sample_8_nu.id_lab.unique()

array(['F25-0008'], dtype=object)

## 3.7 Data corrections

Mauricio detected outliers. He notified Isabel Molina of this via email on 2026_07_22.

According to the review conducted by Isabel Molina on 2026_07_28, the sample with lab ID: F25-2184 will be reprocessed by Johanna, and the results will be included in the next subset. Therefore, it has been decided to remove it.

2026_09_26 Update: The sample with laboratory ID number F25-2184 has been reprocessed in subset 5; its information appears to be correct. For this reason, only the data from subset 5 will be retained from this entry; the remaining records will be deleted.

In [206]:
#Apparently the subset data is on string format, to check this we run:
gas.loc[gas['id_lab'] == 'F25-2184', 'subset'].value_counts(dropna=False)

,count
subset,
5,24
4,8


In [207]:
#Then we delete all F25-2184 records with subset different to 5
gas = gas[~((gas['id_lab'] == 'F25-2184') & (gas['subset'].astype(str) != '5'))]

During a review conducted on 2026_09_26, I detected anomalous methane intensity values in sample F25-2190. I reviewed the raw data files and observed that only three records from Run1 are consistent with the expected values. Therefore, the records from the Runs  have been removed.

In [208]:
#Then we delete all F25-2190 records with run different to 1
gas = gas[~((gas['id_lab'] == 'F25-2190') & (gas['run'].astype(str) != '1'))]

During a review conducted on 2026_09_26, I detected anomalous methane intensity values in sample F25-2193. I reviewed the raw data files and observed that only three records from Run 3  are consistent with the expected values. Therefore, the records from this Run have been removed.

In [209]:
#Then we delete all F25-2193 records with run 3
gas = gas[~((gas['id_lab'] == 'F25-2193') & (gas['run'].astype(str) == '3'))]

During a review conducted on 2026_09_26, I detected that for some samples the functional group was not correctly assigned. Here I solved this issue.

In [210]:
gas.functional_group.unique()

array(['Forage', 'Concentrate', 'Herbaceous_legumes', 'Shrub_Trees', nan,
       'Browse', 'Grass', 'Fabales', 'Green Meal', 'Compound feed'],
      dtype=object)

In [211]:
# Assigning functional group 'Shurb_Trees' for 'Cajanus cajan', because some cells contains 'Forage'
gas.loc[gas['tax_name'] == 'Cajanus cajan', 'functional_group'] = 'Shrub_Trees'

In [212]:
# Assigning functional group 'Shurb_Trees' for 'Desmanthus pernambucanus', because some cells contains 'Browse'
gas.loc[gas['tax_name'] == 'Desmanthus pernambucanus', 'functional_group'] = 'Shrub_Trees'

In [213]:
# Assigning functional group 'Shurb_Trees' for 'Leucaena diversifolia', because some cells contains 'Browse'
gas.loc[gas['tax_name'] == 'Leucaena diversifolia', 'functional_group'] = 'Shrub_Trees'

In [214]:
# Assigning functional group 'Shurb_Trees' for 'Desmodium	cinereum', because some cells contains 'Browse'
gas.loc[gas['tax_name'] == 'Desmodium cinereum', 'functional_group'] = 'Shrub_Trees'

In [177]:
gas[gas['functional_group'] == 'Fabales'].iloc[:,:15].head(50)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,batch,run,replication


In [215]:
gas.functional_group.unique()

array(['Forage', 'Concentrate', 'Herbaceous_legumes', 'Shrub_Trees', nan,
       'Grass', 'Green Meal', 'Compound feed'], dtype=object)

Note: Functional groups different to 'Herbaceous_legumes', 'Shrub_Trees' and 'Grass', correspond to standards or diets.

**Isabel also found errors in the F24 samples F24-3539 to F24-3547. which will be corrected Later, because must be reviewed carefully** Should be decided if correction will be make on site or changing csv input files.


# 4.0 Curation

In [216]:
remarks_columns = ['information_remarks_1','information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
                   'digest_remarks_1', 'digest_remarks_2']

# Convert columns to string type and check for 'elim' or 'descar' (case-insensitive)
conditions = []

for col in remarks_columns:
    condition = gas[col].astype(str).str.contains(
        r'elim|descar',
        case=False,
        na=False,
        regex=True
    )
    conditions.append(condition)

# Combine the remarks conditions using OR logic (any() along axis=1)
# Create a boolean series where True means 'elim' or 'descar' is found in at least one remark column
has_keywords = pd.concat(conditions, axis=1).any(axis=1)

# New condition: check if 'methane_intensity' is NaN
is_methane_intensity_null = gas['methane_intensity'].isna()

# New condition: check if 'undigested_dm_g' is NaN
is_undigested_dm_g_null = gas['undigested_dm_g'].isna()

# Combine all conditions for deletion using OR logic
final_delete_condition = has_keywords | is_methane_intensity_null | is_undigested_dm_g_null
#

# Create the 'delete' column, assigning 'yes' or 'no' based on the final condition
gas['delete'] = np.where(final_delete_condition, 'yes', 'no')

In [217]:
gas_clean = gas[gas['delete'] == 'no']

## 4.1 Save gas and nutrition dataframes

In [218]:
gas_clean.to_csv('/content/drive/MyDrive/lmf/output/2026_09_26_trial_database_curation/gas_clean_complete_subsets_1234561h_2026_09_26.csv', index=None)

In [219]:
nutrition.to_csv('/content/drive/MyDrive/lmf/output/2026_09_26_trial_database_curation/nutrition_complete_subsets_1234561h_2026_09_26.csv', index=None)

# 5.0 Quality Check

## 5.1 Cleaning functions

Number of id_labs by subset in gas dataframe

In [220]:
# Function to clean and standardize lab/sample IDs to FXX-XXXX format
def clean_lab_ids(series):
    return (
        series
        .astype(str)
        .str.replace(r'\s+', '-', regex=True)                 # Replace spaces with -
        .str.replace(r'^(F\d{2})(\d+)', r'\1-\2', regex=True) # Format FXX-XXXX
    )


# Usage:
# subset_1_information_samples['10_ciat_lab_id'] = clean_lab_ids(subset_1_information_samples['10_ciat_lab_id'])

## 5.2 Import data
(Isabel averaged samples)

In [221]:
isa_subset_1 = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/isa_counts/isa_subset_1.csv')
isa_subset_2 = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/isa_counts/isa_subset_2.csv')
isa_subset_3 = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/isa_counts/isa_subset_3.csv')
isa_subset_4 = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/isa_counts/isa_subset_4.csv')

## 5.3 Formating

In [222]:
isa_subset_1['id_lab'] = clean_lab_ids(isa_subset_1['id_lab'])
isa_subset_2['id_lab'] = clean_lab_ids(isa_subset_2['id_lab'])
isa_subset_3['id_lab'] = clean_lab_ids(isa_subset_3['id_lab'])
isa_subset_4['id_lab'] = clean_lab_ids(isa_subset_4['id_lab'])

## 5.4 Database sample counts by subset

In [223]:
subset_1 = gas[gas['subset'] == 1]
subset_1_clean = gas_clean[gas_clean['subset'] == 1]

In [224]:
subset_2 = gas[gas['subset'] == 2]
subset_2_clean = gas_clean[gas_clean['subset'] == 2]

In [225]:
subset_3 = gas[gas['subset'] == 3]
subset_3_clean = gas_clean[gas_clean['subset'] == 3]

In [226]:
subset_4 = gas[gas['subset'] == 4]
subset_4_clean = gas_clean[gas_clean['subset'] == 4]

## 5.5 Comparison between database sample counts and isabel average counts

In [227]:
print('-'*30)
print('     COUNTS BEFORE FILTERING')
print('          Database    Isabel ')
print('Subset 1:', '  ', subset_1.id_lab.nunique(), '     ', isa_subset_1.id_lab.nunique())
print('Subset 2:', '  ', subset_2.id_lab.nunique(), '     ', isa_subset_2.id_lab.nunique() )
print('Subset 3:', '   ', subset_3.id_lab.nunique(), '     ', isa_subset_3.id_lab.nunique() )
print('Subset 4:', '  ', subset_4.id_lab.nunique(), '     ', isa_subset_4.id_lab.nunique() )
print('-'*30)
print('     COUNTS AFTER FILTERING')
print('          Database    Isabel ')
print('Subset 1:', '  ', subset_1_clean.id_lab.nunique(), '     ', isa_subset_1.id_lab.nunique())
print('Subset 2:', '  ', subset_2_clean.id_lab.nunique(), '     ', isa_subset_2.id_lab.nunique() )
print('Subset 3:', '   ', subset_3_clean.id_lab.nunique(), '     ', isa_subset_3.id_lab.nunique() )
print('Subset 4:', '  ', subset_4_clean.id_lab.nunique(), '     ', isa_subset_4.id_lab.nunique() )
print('-'*30)

------------------------------
     COUNTS BEFORE FILTERING
          Database    Isabel 
Subset 1:    0       232
Subset 2:    0       233
Subset 3:     0       128
Subset 4:    0       144
------------------------------
     COUNTS AFTER FILTERING
          Database    Isabel 
Subset 1:    0       232
Subset 2:    0       233
Subset 3:     0       128
Subset 4:    0       144
------------------------------


## 5.6 Duplicates at Isabel average counts

In [228]:
repeated_isa_subset_1 = (isa_subset_1["id_lab"].value_counts().loc[lambda x: x > 1].to_frame("n_replicates").reset_index(names="id_lab"))
repeated_isa_subset_2 = (isa_subset_2["id_lab"].value_counts().loc[lambda x: x > 1].to_frame("n_replicates").reset_index(names="id_lab"))
repeated_isa_subset_3 = (isa_subset_3["id_lab"].value_counts().loc[lambda x: x > 1].to_frame("n_replicates").reset_index(names="id_lab"))
repeated_isa_subset_4 = (isa_subset_4["id_lab"].value_counts().loc[lambda x: x > 1].to_frame("n_replicates").reset_index(names="id_lab"))

In [229]:
print('Repeated samples in Isabel database')
print('             Samples    Repetitions')
print('Subset 1:      ', repeated_isa_subset_1.shape[0], '        ',  repeated_isa_subset_1["n_replicates"].sum())
print('Subset 2:       ', repeated_isa_subset_2.shape[0], '        ',  repeated_isa_subset_2["n_replicates"].sum())
print('Subset 3:       ', repeated_isa_subset_3.shape[0], '        ',  repeated_isa_subset_3["n_replicates"].sum())
print('Subset 4:       ', repeated_isa_subset_4.shape[0], '         ',  repeated_isa_subset_4["n_replicates"].sum())

Repeated samples in Isabel database
             Samples    Repetitions
Subset 1:       36          86
Subset 2:        3          24
Subset 3:        3          11
Subset 4:        1           5


In [230]:
list_repeated_isa_subset_1 = repeated_isa_subset_1.id_lab.to_list()
print('Repeated samples:',len(list_repeated_isa_subset_1))
print(list_repeated_isa_subset_1)

Repeated samples: 36
['F25-0017', 'F25-0018', 'F25-0019', 'F24-3427', 'F24-3431', 'F24-3428', 'F24-3429', 'F24-3430', 'F24-3443', 'F24-3444', 'F24-3445', 'F24-3446', 'F24-3447', 'F24-3448', 'F24-3449', 'F24-3442', 'F24-3434', 'F24-3441', 'F24-3440', 'F24-3439', 'F24-3438', 'F24-3437', 'F24-3436', 'F24-3435', 'F24-3450', 'F24-3453', 'F24-3452', 'F24-3455', 'F24-3451', 'F24-3432', 'F24-3433', 'F24-3426', 'F24-3454', 'F25-0848', 'F25-0845', 'F25-0873']


In [231]:
list_repeated_isa_subset_2 = repeated_isa_subset_2.id_lab.to_list()
print('Repeated samples:',len(list_repeated_isa_subset_2))
print(list_repeated_isa_subset_2)

Repeated samples: 3
['F25-0008', 'F25-0017', 'F25-0018']


In [232]:
list_repeated_isa_subset_3 = repeated_isa_subset_3.id_lab.to_list()
print('Repeated samples:',len(list_repeated_isa_subset_3))
print(list_repeated_isa_subset_3)

Repeated samples: 3
['F25-0008', 'F25-0017', 'F25-1667']


In [233]:
list_repeated_isa_subset_4 = repeated_isa_subset_4.id_lab.to_list()
print('Repeated samples:',len(list_repeated_isa_subset_4))
print(list_repeated_isa_subset_4)

Repeated samples: 1
['F25-0008']


## 5.7 Absent samples

In [234]:
list_subset_1 = subset_1["id_lab"].unique().tolist()
list_subset_2 = subset_2["id_lab"].unique().tolist()
list_subset_3 = subset_3["id_lab"].unique().tolist()
list_subset_4 = subset_4["id_lab"].unique().tolist()

In [235]:
print('Samples subset 1:', len(subset_1["id_lab"].unique()))
print('Samples subset 2:', len(subset_2["id_lab"].unique()))
print('Samples subset 3:', len(subset_3["id_lab"].unique()))
print('Samples subset 4:', len(subset_4["id_lab"].unique()))


Samples subset 1: 0
Samples subset 2: 0
Samples subset 3: 0
Samples subset 4: 0


In [236]:
gas.functional_group.unique()

array(['Forage', 'Concentrate', 'Herbaceous_legumes', 'Shrub_Trees', nan,
       'Grass', 'Green Meal', 'Compound feed'], dtype=object)

In [237]:
list_isa_subset_1 = isa_subset_1["id_lab"].unique().tolist()
list_isa_subset_2 = isa_subset_2["id_lab"].unique().tolist()
list_isa_subset_3 = isa_subset_3["id_lab"].unique().tolist()
list_isa_subset_4 = isa_subset_4["id_lab"].unique().tolist()

In [238]:
absent_in_subset_1 = list(set(list_isa_subset_1) - set(list_subset_1))
print(len(absent_in_subset_1))
print(absent_in_subset_1)

232
['F24-3607', 'F24-3594', 'F24-3445', 'F24-3431', 'F24-3584', 'F24-3514', 'F24-3585', 'F25-0907', 'F24-3486', 'F24-3606', 'F25-0003', 'F24-3498', 'F24-3533', 'F24-3590', 'F24-3483', 'F25-0861', 'F24-3597', 'F24-3609', 'F24-3591', 'F24-3480', 'F24-3439', 'F25-0891', 'F25-1012', 'F24-3523', 'F25-0773', 'F24-3588', 'F24-3503', 'F24-3459', 'F25-0914', 'F24-3535', 'F24-3592', 'F25-0924', 'F24-1075', 'F24-3526', 'F24-3454', 'F24-3595', 'F24-3422', 'F24-3438', 'F24-3519', 'F24-3429', 'F25-0001', 'F24-3587', 'F25-0869', 'F24-3468', 'F25-0897', 'F24-3433', 'F25-0002', 'F24-3443', 'F25-0772', 'F25-1001', 'F24-3608', 'F24-3473', 'F24-3489', 'F25-1000', 'F24-3446', 'F24-3499', 'F25-0899', 'F24-3476', 'F24-3507', 'F24-3440', 'F25-0908', 'F25-0820', 'F25-0999', 'F24-3525', 'F24-3493', 'F24-3450', 'F24-3465', 'F24-3497', 'F25-0783', 'F24-1047', 'F24-3513', 'F24-3442', 'F24-3520', 'F25-1007', 'F24-3494', 'F25-0992', 'F24-3604', 'F24-3426', 'F25-1004', 'F24-3479', 'F24-3428', 'F24-3512', 'F24-3416',

In [239]:
absent_in_subset_2 = list(set(list_isa_subset_2) - set(list_subset_2))
print(len(absent_in_subset_2))
print(absent_in_subset_2)
#Estas son las que se eliminan porque estan duplicadas en el subset 2 y 3

233
['F25-0964', 'F24-3660', 'F25-2182', 'F25-2095', 'F24-3613', 'F25-2084', 'F25-1704', 'F25-0962', 'F25-2091', 'F25-2117', 'F25-2120', 'F24-3635', 'F24-3544', 'F24-3537', 'F25-2079', 'F25-2163', 'F25-1707', 'F25-1721', 'F25-0975', 'F25-2149', 'F25-2145', 'F25-0978', 'F25-2089', 'F25-2088', 'F25-0989', 'F25-0968', 'F25-1724', 'F24-3614', 'F25-0969', 'F25-2092', 'F24-3638', 'F24-3631', 'F25-0945', 'F25-2097', 'F25-2100', 'F25-1664', 'F25-2156', 'F24-3648', 'F24-3628', 'F25-2080', 'F25-1718', 'F25-0929', 'F25-1653', 'F25-1711', 'F25-0985', 'F25-1703', 'F25-2108', 'F25-0937', 'F24-3658', 'F25-2177', 'F25-0984', 'F24-3656', 'F24-3618', 'F25-2111', 'F24-3542', 'F25-2147', 'F25-2110', 'F25-0981', 'F24-3641', 'F25-2093', 'F25-1708', 'F25-1712', 'F24-3657', 'F24-3546', 'F24-3654', 'F25-1722', 'F24-3615', 'F25-2172', 'F24-3640', 'F25-0947', 'F24-3652', 'F25-0933', 'F25-1714', 'F24-3650', 'F25-1725', 'F25-2082', 'F24-3543', 'F25-0938', 'F25-0949', 'F25-2102', 'F24-3646', 'F25-2121', 'F25-0941',

In [240]:
absent_in_subset_3 = list(set(list_isa_subset_3) - set(list_subset_3))
print(len(absent_in_subset_3))
print(absent_in_subset_3)

128
['F25-1666', 'F25-2632', 'F24-3575', 'F24-3576', 'F25-2566', 'F25-1686', 'F24-3558', 'F25-1700', 'F25-2625', 'F25-0974', 'F25-2171', 'F25-2624', 'F25-2631', 'F25-1663', 'F25-2629', 'F24-3570', 'F24-3569', 'F25-1662', 'F25-2627', 'F25-1691', 'F25-1661', 'F25-1679', 'F25-1664', 'F24-3562', 'F24-3551', 'F25-1653', 'F24-3579', 'F25-2638', 'F25-1682', 'F24-3580', 'F24-3553', 'F25-2170', 'F25-1670', 'F25-2131', 'F24-3578', 'F25-1696', 'F24-3571', 'F25-1688', 'F24-3574', 'F25-1674', 'F24-3573', 'F24-3559', 'F25-2567', 'F24-3564', 'F25-0980', 'F25-1669', 'F25-1681', 'F25-2630', 'F25-1698', 'F25-2141', 'F25-2636', 'F25-1678', 'F25-2639', 'F25-0971', 'F24-3549', 'F24-3552', 'F24-3550', 'F24-3568', 'F25-0954', 'F25-1667', 'F25-2637', 'Dieta-1_Exp1', 'F24-3572', 'F25-1684', 'F25-2140', 'F25-1699', 'F25-1654', 'F24-3581', 'Dieta-2_Exp1', 'F25-2628', 'F25-2633', 'F25-1683', 'F25-1660', 'F25-2134', 'F25-1656', 'F24-3554', 'F25-1657', 'F24-3566', 'F25-1685', 'F25-1658', 'F25-2626', 'F25-2139', 'F2

In [241]:
absent_in_subset_4 = list(set(list_isa_subset_4) - set(list_subset_4))
print(len(absent_in_subset_4))
print(absent_in_subset_4)

144
['F25-1768', 'F25-2598', 'F25-2186', 'F25-1765', 'F25-1775', 'F25-2184', 'F25-2610', 'F25-2192', 'F26-0038', 'F26-0030', 'F26-0023', 'F26-0040', 'F26-0041', 'F25-2166', 'F25-1754', 'F25-1756', 'F25-2581', 'F25-2600', 'F26-0024', 'F25-1752', 'F25-2577', 'F25-1760', 'F25-2603', 'F25-2191', 'F25-2623', 'F25-2595', 'F25-2586', 'F26-0022', 'F25-1735', 'F25-1762', 'F25-2104', 'F26-0021', 'F25-2188', 'F25-2615', 'F25-1730', 'F25-2576', 'F25-2601', 'F26-0027', 'F25-1745', 'F25-1763', 'F25-1774', 'F25-2585', 'F25-2619', 'F25-2640', 'F26-0017', 'F26-0015', 'F25-2616', 'F25-1764', 'F26-0018', 'F25-2617', 'F25-2582', 'F25-2584', 'F25-1767', 'F25-1772', 'F25-1759', 'F25-2596', 'F25-2189', 'F25-1749', 'F25-1770', 'F25-1769', 'F25-2621', 'F25-2590', 'F26-0035', 'F25-1748', 'F25-1737', 'F25-2611', 'F25-2594', 'F25-2578', 'F26-0039', 'F25-2620', 'F26-0032', 'F25-2185', 'F25-2607', 'F26-0020', 'F25-1750', 'F25-1741', 'F26-0033', 'F25-2605', 'F25-2613', 'F26-0016', 'F25-2604', 'F26-0019', 'F25-2602',

## 5.8 MANUAL CHECK ON ISABEL FILES:

### Subset 1


F25-0772: Tab 3.1 present (Only Run 1) - Tab 4.0  absent - Tab 4.1av present - id: ICARDA-165563 - tax_name: Trifolium pratense

F25-0825: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-6984-4-119709 - tax_name: Medicago sativa

F25-0913: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: **CIAT-1853** - tax_name: Stylosanthes guianensis **Absent in other subsets**

F24-1047: Tab 3.1 present (**Extrange tddm**) - Tab 4.0 absent - Tab 4.1av present - id: LMF - tax_name: Albizzia saman

F25-0820: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ICARDA-T216010

F24-1084: Tab 3.1 present (**Extrange tddm**) - Tab 4.0 absent - Tab 4.1av present - id: LMF - tax_name: Brachiaria decumbens

F25-0839: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-7049-18-119723 - tax_name: Trifolium repens

F25-0837: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id:ILRI-10920-16-119721 - tax_name: Arachis pintoi

F25-0861: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-14539-40-119745 - tax_name: Centrosema virginianum

F25-0823: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-6529-2-119707 - tax_name: Labalab purpureues

F24-1075: Tab 3.1 present (**Extrange tddm**) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-6529-2-119707 - tax_name: Labalab purpureues

F25-0926: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: **CIAT-6818** - tax_name: Panicum maximum **Absent in other subsets**

F25-0824: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-10925-3-119708 - tax_name: Arachis glabrata


F25-0904: Tab 3.1 present (Only Run 1) - Tab 4.0 **present** - Tab 4.1av present - id: **CIAT-22752** - tax_name: Labalab purpureues **Absent in other subsets**

F25-0868: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-70-47-120164 - tax_name: Leucaena leucocephala


F25-0813: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ICARDA-T216003 - tax_name: Rhus tripartita (Ucria) Grande

F24-1095: Tab 3.1 present (**Extrange tddm**) - Tab 4.0 absent - Tab 4.1av present - id: LMF - tax_name: Setaria sphacelata

F25-0908: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: **CIAT-22192** - tax_name: Leucaena diversifolia **Absent in other subsets**

F24-1070: Tab 3.1 present (**Extrange tddm**) - Tab 4.0 absent - Tab 4.1av present - id: LMF - tax_name: Gliricidia sepium

F25-0914: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: **CIAT-10625** - tax_name: Stylosanthes scabra **Absent in other subsets**


F25-0788: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ICARDA-J13 - tax_name: Cichorium pumilum


F25-0783: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ICARDA-J8 - tax_name: Daucus carota

F25-0784: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ICARDA-J8 - tax_name: Daucus carota



### Subset 2

['F25-1652', 'F25-1658', 'F25-1665', 'F25-1655', 'F25-1651', 'F25-1654', 'F25-1656', 'F25-1653', 'F25-1664'] Deleated because were reprocessed on subset 3





## 5.9 Revision of Isabel selection on Subset 4

# 6.0 Gas

## 6.1 Mean

In [242]:
gas_clean.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete'],
      dtype='object')

In [243]:
#Columns processing
category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete']
numeric_columns = [ 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm']

for df in [gas_clean]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

/tmp/ipykernel_20878/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype('category')
/tmp/ipykernel_20878/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype('category')
/tmp/ipykernel_20878/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stab

In [244]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_gas'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [245]:
gas_clean_av = mean_with_replicates(gas_clean, category_columns, numeric_columns)

/tmp/ipykernel_20878/1161671181.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_20878/1161671181.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_20878/1161671181.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [246]:
gas_clean_av2 = gas_clean_av[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'net_gas_8h_ml','net_gas_24h_ml','gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas']]

In [247]:
gas_clean_av2

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h,n_replicates_gas
0,Dieta-1-:-F263137-+-F261072-(40%/60%)-Estrella...,Dieta-1-:-F-263137-+-F-261072-(40%/60%)-Estrel...,5,1777,ISABEL MOLINA/ EXP Camaras In vivo,NaN,NaN,29.02,59.27,125.28,35.69,2.85,12.68,13.80,17.29,48.82,1.91,6
1,Dieta-1_Exp1,Dieta-1-Exp-1,3,839,LMF-invivo,Dieta 1_Exp1,NaN,30.67,67.46,149.92,34.90,2.31,14.42,14.69,21.82,71.84,321.88,6
2,Dieta-1_Exp2,Dieta-1-Exp-2,3,1019,LMF-invivo,Dieta 1_Exp2,NaN,28.26,57.51,129.22,33.33,2.41,13.66,14.43,18.49,70.34,313.12,8
3,Dieta-2-:-F263137-+-F263138-(55%/45%)-Estrella...,Dieta-2-:-F-263137-+-F-263138-(55%/45%)-Estrel...,5,1779,ISABEL MOLINA/ EXP Camaras In vivo,NaN,NaN,35.36,74.77,158.74,42.41,2.68,16.03,16.99,26.98,63.70,0.90,6
4,Dieta-2_Exp1,Dieta-2-Exp-1,3,738,LMF-invivo,Dieta 2_Exp1,NaN,29.60,62.15,139.52,30.16,2.16,13.87,14.36,20.02,71.36,317.83,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,F26-4733,CIAT-15551,6,1545,Genetic_bank,Centrosema molle,Herbaceous_legumes,36.17,68.69,141.71,39.70,2.81,15.98,16.78,23.77,61.36,-0.87,6
1211,F26-4734,CIAT-17009,6,1547,Genetic_bank,Canavalia brasiliensis,Herbaceous_legumes,42.25,83.08,171.40,55.04,3.22,15.40,17.73,30.41,55.53,1.10,6
1212,F26-4735,CIAT-17462,6,1549,Genetic_bank,Canavalia brasiliensis,Herbaceous_legumes,37.42,79.66,164.48,54.20,3.29,15.18,17.43,28.68,54.60,5.62,6
1213,F26-4736,CIAT-17009,6,1551,Genetic_bank,Canavalia brasiliensis,Herbaceous_legumes,37.00,77.99,164.40,57.99,3.53,15.17,16.66,27.39,47.53,-3.64,6


In [248]:
gas_clean_av2.to_csv('/content/drive/MyDrive/lmf/output/2026_09_26_trial_database_curation/gas_clean_average_subsets_1234561h_2026_09_26.csv', index=None)

## 6.2 Standard deviation

In [249]:
# Group by subset and id_lab, keeping categorical columns
def std_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the standard deviation of the numeric columns
    std_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .std()
          .round(2)
    )

    # Count the number of replicates in each group
    std_df['n_replicates_gas'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, stds, and replicate counts
    result = (
        cat_df
        .join(std_df)
        .reset_index()
    )

    return result

In [250]:
gas_clean_std = std_with_replicates(gas_clean, category_columns, numeric_columns)

/tmp/ipykernel_20878/2013314210.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_20878/2013314210.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_20878/2013314210.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [251]:
gas_clean_std2 = gas_clean_std[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'net_gas_8h_ml','net_gas_24h_ml','gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas']]

In [252]:
gas_clean_std2.to_csv('/content/drive/MyDrive/lmf/output/2026_09_26_trial_database_curation/gas_clean_std_subsets_1234561h_2026_09_26.csv', index=None)

#7.0 Nutrition

## 7.1 Mean

In [253]:
nutrition.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm'],
      dtype='object')

In [254]:
#Columns processing

category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat']
numeric_columns = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']

for df in [nutrition]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [255]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_nutrition'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [256]:
nutrition_av = mean_with_replicates(nutrition, category_columns, numeric_columns)

/tmp/ipykernel_20878/1194620962.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_20878/1194620962.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_20878/1194620962.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [257]:
nutrition_av.columns

Index(['id_lab', 'subset', 'no', 'requisitioner', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition'],
      dtype='object')

In [258]:
nutrition_av2 = nutrition_av[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition']]

In [259]:
nutrition_av2.head()

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
0,Dieta-1-:-F263137-+-F261072-(40%/60%)-Estrella...,Dieta-1-:-F-263137-+-F-261072-(40%/60%)-Estrel...,5,NaN,ISABEL MOLINA/ EXP Camaras In vivo,NaN,NaN,94.60,8.97,91.03,20.42,35.20,62.75,2
1,Dieta-1_Exp1,Dieta-1-Exp-1,3,193.0,LMF-invivo,Dieta 1_Exp1,NaN,89.00,9.50,89.53,17.68,35.48,71.52,2
2,Dieta-1_Exp2,Dieta-1-Exp-2,3,235.0,LMF-invivo,Dieta 1_Exp2,NaN,89.00,9.50,89.53,0.00,0.00,0.00,2
3,Dieta-2-:-F263137-+-F263138-(55%/45%)-Estrella...,Dieta-2-:-F-263137-+-F-263138-(55%/45%)-Estrel...,5,NaN,ISABEL MOLINA/ EXP Camaras In vivo,NaN,NaN,94.15,11.04,88.96,13.32,35.56,67.59,2
4,Dieta-2_Exp1,Dieta-2-Exp-1,3,195.0,LMF-invivo,Dieta 2_Exp1,NaN,89.00,9.10,89.58,13.79,33.91,67.86,2


In [260]:
nutrition_av2.to_csv('/content/drive/MyDrive/lmf/output/2026_09_26_trial_database_curation/nutrition_average_1234561h_2026_09_26.csv', index=None)

## 7.2 Standard deviation

In [261]:
# Group by subset and id_lab, keeping categorical columns
def std_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the standard deviation of the numeric columns
    std_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .std()
          .round(2)
    )

    # Count the number of replicates in each group
    std_df['n_replicates_nutrition'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, stds, and replicate counts
    result = (
        cat_df
        .join(std_df)
        .reset_index()
    )

    return result

In [262]:
nutrition_std = std_with_replicates(nutrition, category_columns, numeric_columns)

/tmp/ipykernel_20878/1995383719.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_20878/1995383719.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_20878/1995383719.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [263]:
nutrition_std2 = nutrition_std[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition']]

In [264]:
nutrition_std2

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
0,Dieta-1-:-F263137-+-F261072-(40%/60%)-Estrella...,Dieta-1-:-F-263137-+-F-261072-(40%/60%)-Estrel...,5,NaN,ISABEL MOLINA/ EXP Camaras In vivo,NaN,NaN,0.03,0.27,0.27,0.20,0.78,1.24,2
1,Dieta-1_Exp1,Dieta-1-Exp-1,3,193.0,LMF-invivo,Dieta 1_Exp1,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
2,Dieta-1_Exp2,Dieta-1-Exp-2,3,235.0,LMF-invivo,Dieta 1_Exp2,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
3,Dieta-2-:-F263137-+-F263138-(55%/45%)-Estrella...,Dieta-2-:-F-263137-+-F-263138-(55%/45%)-Estrel...,5,NaN,ISABEL MOLINA/ EXP Camaras In vivo,NaN,NaN,0.03,0.10,0.10,0.09,0.23,1.37,2
4,Dieta-2_Exp1,Dieta-2-Exp-1,3,195.0,LMF-invivo,Dieta 2_Exp1,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1198,F26-4733,CIAT-15551,6,NaN,Genetic_bank,Centrosema molle,Herbaceous_legumes,0.10,0.04,0.04,0.13,0.14,0.76,2
1199,F26-4734,CIAT-17009,6,NaN,Genetic_bank,Canavalia brasiliensis,Herbaceous_legumes,0.16,0.09,0.09,0.34,0.21,0.57,2
1200,F26-4735,CIAT-17462,6,NaN,Genetic_bank,Canavalia brasiliensis,Herbaceous_legumes,0.01,0.23,0.23,0.41,1.11,1.00,2
1201,F26-4736,CIAT-17009,6,NaN,Genetic_bank,Canavalia brasiliensis,Herbaceous_legumes,0.14,0.08,0.08,0.27,1.05,0.36,2


In [265]:
nutrition_std2.to_csv('/content/drive/MyDrive/lmf/output/2026_09_26_trial_database_curation/nutrition_std_1234561h_2026_09_26.csv', index=None)